# Pipeline final Taller 1

Este notebook reconstruye la solución final del Proyecto 1 - Bosón de Higgs desde los datos crudos de Kaggle hasta el archivo de submission. La idea es que pueda ejecutarse de forma independiente: si `training.csv` y `test.csv` no están en este mismo directorio, el notebook descarga los datos, los extrae, entrena el modelo y genera `pipeline_final_taller_1.csv`.

## Credenciales y datos de Kaggle

Para descargar los datos se necesita el archivo `kaggle.json`, que se obtiene desde la cuenta de Kaggle en `Account > API > Create New Token`. Si el archivo no está junto al notebook, se solicita durante la ejecución: en Colab aparece un selector de archivos y en Jupyter local se pide escribir la ruta donde está guardado. Los archivos descargados y extraídos quedan en el mismo directorio del notebook.

## Librerías utilizadas

Se instalan y cargan las herramientas necesarias para manipular tablas, definir transformaciones, entrenar modelos y descargar los datos desde Kaggle.

In [ ]:
!pip install numpy pandas scikit-learn xgboost kaggle

import subprocess
import sys

import json
import os
import shutil
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import RobustScaler
from xgboost import XGBClassifier

## Configuración fija del experimento final

En esta celda se fijan las decisiones que definen la submission final: la semilla de reproducibilidad, la política de decisión, las columnas físicas esperadas del dataset y los hiperparámetros del XGBoost final.

La columna `Weight` no se usa como variable de entrada del modelo. Su rol aparece más adelante, cuando se utiliza como peso de entrenamiento para que el ajuste esté más alineado con la métrica AMS del desafío.

In [ ]:
SEED = 42
N_SPLITS = 5
VALID_SIZE = 0.1
MISSING_MARKER = -999.0
EPS = 1e-6
GLOBAL_OOF_OPTIMAL_THRESHOLD = 0.06402307003736496
EXPECTED_TEST_ROWS = 550_000
REFERENCE_SIGNAL_PERCENT = 14.06581818181818

SPECIAL_COLUMNS = ["EventId", "Weight", "Label", "KaggleSet", "KaggleWeight"]
JET_NUM_COLUMN = "PRI_jet_num"
JET_NUM_VALUES = [0, 1, 2, 3]
FIXED_JET_ONE_HOT_COLUMNS = [f"PRI_jet_num_{value}" for value in JET_NUM_VALUES]

ALLOWED_FEATURE_COLUMNS = [
    "DER_mass_MMC",
    "DER_mass_transverse_met_lep",
    "DER_mass_vis",
    "DER_pt_h",
    "DER_deltaeta_jet_jet",
    "DER_mass_jet_jet",
    "DER_prodeta_jet_jet",
    "DER_deltar_tau_lep",
    "DER_pt_tot",
    "DER_sum_pt",
    "DER_pt_ratio_lep_tau",
    "DER_met_phi_centrality",
    "DER_lep_eta_centrality",
    "PRI_tau_pt",
    "PRI_tau_eta",
    "PRI_tau_phi",
    "PRI_lep_pt",
    "PRI_lep_eta",
    "PRI_lep_phi",
    "PRI_met",
    "PRI_met_phi",
    "PRI_met_sumet",
    "PRI_jet_num",
    "PRI_jet_leading_pt",
    "PRI_jet_leading_eta",
    "PRI_jet_leading_phi",
    "PRI_jet_subleading_pt",
    "PRI_jet_subleading_eta",
    "PRI_jet_subleading_phi",
    "PRI_jet_all_pt",
]

BASE_LOG_COLUMNS = [
    "DER_mass_MMC",
    "DER_mass_transverse_met_lep",
    "DER_mass_vis",
    "DER_pt_h",
    "DER_deltaeta_jet_jet",
    "DER_mass_jet_jet",
    "DER_deltar_tau_lep",
    "DER_pt_tot",
    "DER_sum_pt",
    "DER_pt_ratio_lep_tau",
    "PRI_tau_pt",
    "PRI_lep_pt",
    "PRI_met",
    "PRI_met_sumet",
    "PRI_jet_leading_pt",
    "PRI_jet_subleading_pt",
    "PRI_jet_all_pt",
]
P04_ENGINEERED_LOG_COLUMNS = ["FE_tau_lep_pt_sum"]

FINAL_XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": SEED,
    "tree_method": "hist",
    "n_jobs": -1,
    "missing": np.nan,
    "learning_rate": 0.03,
    "n_estimators": 800,
    "max_depth": 6,
    "min_child_weight": 1,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "scale_pos_weight": 1.5346,
    "gamma": 0.0,
    "max_delta_step": 0,
    "reg_alpha": 0.5,
    "reg_lambda": 2.0,
}

REFERENCE_KAGGLE = {
    "submission": "Final_global_oof_optimal_threshold.csv",
    "decision_threshold": GLOBAL_OOF_OPTIMAL_THRESHOLD,
    "public_score": 3.61216,
    "private_score": 3.68860,
}

COMPETITION = "higgs-boson"
SUBMISSION_FILENAME = "pipeline_final_taller_1.csv"
SCORES_FILENAME = "pipeline_final_taller_1_scores.csv"
METADATA_FILENAME = "pipeline_final_taller_1_metadata.json"

## Preparación de datos crudos e ingeniería física

Este bloque reúne funciones auxiliares para obtener los datos y construir las variables derivadas del pipeline `P04`. Si faltan `training.csv` o `test.csv`, se descarga la competencia desde Kaggle y se extraen los archivos en el directorio actual.

Luego se definen variables físicas adicionales. Estas variables resumen relaciones entre objetos del evento, por ejemplo separaciones angulares, balances de momento, razones de masa y proporciones asociadas a jets. El objetivo no es cambiar la física del problema, sino hacer explícitas relaciones que el modelo puede aprovechar mejor.

In [ ]:
def notebook_work_dir() -> Path:
    """Returns the directory used by the notebook as its working folder."""
    return Path.cwd().resolve()


def request_kaggle_json(work_dir: Path) -> Path:
    """Requests kaggle.json and stores it next to the notebook."""
    destination = work_dir / "kaggle.json"
    print("No se encontró kaggle.json junto al notebook.")
    print("Descargue el token desde Kaggle: Account > API > Create New Token.")

    try:
        from google.colab import files  # type: ignore[import-not-found]
    except ImportError:
        raw_path = input("Ingrese la ruta completa al archivo kaggle.json: ").strip().strip('"').strip("'")
        if not raw_path:
            raise FileNotFoundError("No se indicó ninguna ruta para kaggle.json.")
        source = Path(raw_path).expanduser().resolve()
        if not source.exists():
            raise FileNotFoundError(f"No se encontró el archivo indicado: {source}")
        if source != destination.resolve():
            shutil.copy2(source, destination)
    else:
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise FileNotFoundError("Debe subir un archivo llamado kaggle.json.")
        destination.write_bytes(uploaded["kaggle.json"])

    return destination


def configure_kaggle_credentials(work_dir: Path) -> None:
    """Configures Kaggle credentials from kaggle.json next to the notebook."""
    kaggle_file = work_dir / "kaggle.json"
    if not kaggle_file.exists():
        kaggle_file = request_kaggle_json(work_dir)

    if os.name != "nt":
        kaggle_file.chmod(0o600)
    os.environ["KAGGLE_CONFIG_DIR"] = str(work_dir.resolve())
    print(f"Usando credenciales de Kaggle en {kaggle_file.name}")


def run_command(command: list[str]) -> None:
    """Runs a command and raises a clear error if it fails."""
    completed = subprocess.run(command, check=False, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        if completed.stderr:
            print(completed.stderr)
        raise RuntimeError(f"Falló el comando: {' '.join(command)}")


def extract_zip(zip_path: Path, output_dir: Path) -> None:
    """Extracts a zip file into output_dir."""
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(output_dir)


def ensure_higgs_data(work_dir: Path) -> tuple[Path, Path]:
    """Downloads and extracts Higgs data when training.csv or test.csv are missing."""
    data_dir = work_dir

    train_path = data_dir / "training.csv"
    test_path = data_dir / "test.csv"
    if train_path.exists() and test_path.exists():
        print(f"Datos encontrados en {data_dir.resolve()}")
        return train_path, test_path

    configure_kaggle_credentials(work_dir)

    competition_zip = data_dir / f"{COMPETITION}.zip"
    training_zip = data_dir / "training.zip"
    test_zip = data_dir / "test.zip"

    if not competition_zip.exists() and not (training_zip.exists() and test_zip.exists()):
        print(f"Descargando datos de Kaggle: {COMPETITION}")
        kaggle_executable = shutil.which("kaggle")
        if kaggle_executable is None:
            raise RuntimeError("No se encontró el ejecutable kaggle. Instale la librería con: pip install kaggle")
        run_command([
            kaggle_executable,
            "competitions",
            "download",
            "-c",
            COMPETITION,
            "-p",
            str(data_dir),
        ])

    if competition_zip.exists() and not (training_zip.exists() and test_zip.exists()):
        print(f"Extrayendo {competition_zip.name}")
        extract_zip(competition_zip, data_dir)

    if not train_path.exists():
        if not training_zip.exists():
            raise FileNotFoundError(f"No se encontró {training_zip}")
        print(f"Extrayendo {training_zip.name}")
        extract_zip(training_zip, data_dir)

    if not test_path.exists():
        if not test_zip.exists():
            raise FileNotFoundError(f"No se encontró {test_zip}")
        print(f"Extrayendo {test_zip.name}")
        extract_zip(test_zip, data_dir)

    if not train_path.exists() or not test_path.exists():
        raise FileNotFoundError("No se pudieron preparar training.csv y test.csv.")

    print(f"Datos listos en {data_dir.resolve()}")
    return train_path, test_path


def abs_delta_phi(phi_a: pd.Series, phi_b: pd.Series) -> pd.Series:
    return ((phi_a - phi_b + np.pi) % (2 * np.pi) - np.pi).abs()


def add_p04_features(X: pd.DataFrame) -> pd.DataFrame:
    result = X.copy()
    result["FE_abs_dphi_tau_lep"] = abs_delta_phi(result["PRI_tau_phi"], result["PRI_lep_phi"])
    result["FE_abs_dphi_tau_met"] = abs_delta_phi(result["PRI_tau_phi"], result["PRI_met_phi"])
    result["FE_abs_dphi_lep_met"] = abs_delta_phi(result["PRI_lep_phi"], result["PRI_met_phi"])
    result["FE_abs_deta_tau_lep"] = (result["PRI_tau_eta"] - result["PRI_lep_eta"]).abs()
    result["FE_tau_lep_pt_sum"] = result["PRI_tau_pt"] + result["PRI_lep_pt"]
    result["FE_tau_pt_fraction"] = result["PRI_tau_pt"] / (result["PRI_tau_pt"] + result["PRI_lep_pt"] + EPS)
    result["FE_met_over_tau_lep_pt"] = result["PRI_met"] / (result["PRI_tau_pt"] + result["PRI_lep_pt"] + EPS)
    result["FE_met_over_sumet"] = result["PRI_met"] / (result["PRI_met_sumet"] + EPS)
    result["FE_pt_balance"] = result["DER_pt_tot"] / (result["DER_sum_pt"] + EPS)
    result["FE_mmc_over_visible_mass"] = result["DER_mass_MMC"] / (result["DER_mass_vis"] + EPS)
    result["FE_transverse_mass_over_visible_mass"] = result["DER_mass_transverse_met_lep"] / (result["DER_mass_vis"] + EPS)
    result["FE_jet_all_pt_fraction"] = result["PRI_jet_all_pt"] / (result["DER_sum_pt"] + EPS)
    result["FE_leading_jet_pt_fraction"] = result["PRI_jet_leading_pt"] / (result["PRI_jet_all_pt"] + EPS)
    result["FE_subleading_jet_pt_fraction"] = result["PRI_jet_subleading_pt"] / (result["PRI_jet_all_pt"] + EPS)
    result["FE_jet_pt_balance"] = (result["PRI_jet_leading_pt"] - result["PRI_jet_subleading_pt"]) / (
        result["PRI_jet_leading_pt"] + result["PRI_jet_subleading_pt"] + EPS
    )
    result["FE_abs_dphi_leading_subleading_jet"] = abs_delta_phi(
        result["PRI_jet_leading_phi"],
        result["PRI_jet_subleading_phi"],
    )
    result["FE_mass_jet_over_sum_pt"] = result["DER_mass_jet_jet"] / (result["DER_sum_pt"] + EPS)
    return result


def validate_raw_higgs(train_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
    feature_cols = [column for column in train_df.columns if column not in SPECIAL_COLUMNS]
    test_feature_cols = [column for column in test_df.columns if column not in SPECIAL_COLUMNS]
    if feature_cols != ALLOWED_FEATURE_COLUMNS:
        raise ValueError("Las features de training.csv no coinciden con la especificación esperada.")
    if test_feature_cols != feature_cols:
        raise ValueError("Las features de training.csv y test.csv no coinciden.")
    invalid_labels = sorted(set(train_df["Label"].dropna().unique()) - {"b", "s"})
    if invalid_labels:
        raise ValueError(f"Labels inválidos: {invalid_labels}")

## Preprocesamiento P04

El preprocesamiento transforma los datos crudos en una matriz numérica estable para el modelo. Primero se agregan las variables físicas derivadas, luego se crean indicadores de valores faltantes, se codifica `PRI_jet_num` como variable categórica y se aplican transformaciones logarítmicas a magnitudes positivas con colas largas.

La imputación y el escalado se ajustan solo sobre datos de entrenamiento. Esto evita usar información del conjunto que el modelo no debería conocer al momento de aprender.

In [ ]:
class P04Preprocessor(BaseEstimator, TransformerMixin):
    def fit(self, X: pd.DataFrame, y: pd.Series | None = None) -> "P04Preprocessor":
        prepared = self._prepare_features(X)
        self.missing_indicator_cols_ = [
            column for column in prepared.columns
            if column != JET_NUM_COLUMN and prepared[column].isna().any()
        ]
        continuous = prepared.drop(columns=[JET_NUM_COLUMN])
        self.continuous_cols_ = continuous.columns.tolist()
        self.log_cols_ = [
            column for column in BASE_LOG_COLUMNS + P04_ENGINEERED_LOG_COLUMNS
            if column in self.continuous_cols_
        ]
        logged = self._apply_log1p(continuous)
        self.imputer_ = SimpleImputer(strategy="median")
        imputed = self.imputer_.fit_transform(logged[self.continuous_cols_])
        self.scaler_ = RobustScaler(with_centering=True, with_scaling=True, quantile_range=(25.0, 75.0))
        self.scaler_.fit(imputed)
        self.final_feature_order_ = (
            self.continuous_cols_
            + [f"is_missing_{column}" for column in self.missing_indicator_cols_]
            + FIXED_JET_ONE_HOT_COLUMNS
        )
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        prepared = self._prepare_features(X)
        indicator_frame = pd.DataFrame(index=prepared.index)
        for column in self.missing_indicator_cols_:
            indicator_frame[f"is_missing_{column}"] = prepared[column].isna().astype(np.float32)

        jet_one_hot = pd.DataFrame(
            {
                f"PRI_jet_num_{value}": (prepared[JET_NUM_COLUMN].astype(int) == value).astype(np.float32)
                for value in JET_NUM_VALUES
            },
            index=prepared.index,
        )
        continuous = prepared.drop(columns=[JET_NUM_COLUMN])[self.continuous_cols_]
        logged = self._apply_log1p(continuous)
        imputed = self.imputer_.transform(logged[self.continuous_cols_])
        scaled = self.scaler_.transform(imputed)
        scaled_frame = pd.DataFrame(scaled, columns=self.continuous_cols_, index=prepared.index)
        result = pd.concat([scaled_frame, indicator_frame, jet_one_hot], axis=1)
        return result[self.final_feature_order_].astype(np.float32)

    def _prepare_features(self, X: pd.DataFrame) -> pd.DataFrame:
        if JET_NUM_COLUMN not in X.columns:
            raise ValueError("PRI_jet_num no existe antes del preprocesamiento.")
        return add_p04_features(X)

    def _apply_log1p(self, X: pd.DataFrame) -> pd.DataFrame:
        result = X.copy()
        for column in self.log_cols_:
            values = result[column].dropna()
            if (values < 0).any():
                raise ValueError(f"{column} tiene valores negativos y no admite log1p.")
            result[column] = np.log1p(result[column])
        return result


class CVXGBoostEnsemble(BaseEstimator, ClassifierMixin):
    def __init__(self, params: dict[str, object], n_splits: int = N_SPLITS, seed: int = SEED) -> None:
        self.params = params
        self.n_splits = n_splits
        self.seed = seed

    def fit(self, X: pd.DataFrame, y: pd.Series, sample_weight: pd.Series | None = None) -> "CVXGBoostEnsemble":
        self.models_ = []
        self.fit_seconds_ = []
        cv = StratifiedKFold(n_splits=self.n_splits, shuffle=True, random_state=self.seed)
        for fold_id, (train_idx, _) in enumerate(cv.split(X, y), start=1):
            model = XGBClassifier(**self.params)
            start = time.perf_counter()
            fit_kwargs = {}
            if sample_weight is not None:
                fit_kwargs["sample_weight"] = sample_weight.iloc[train_idx].reset_index(drop=True)
            model.fit(
                X.iloc[train_idx].reset_index(drop=True),
                y.iloc[train_idx].reset_index(drop=True),
                **fit_kwargs,
            )
            self.fit_seconds_.append(time.perf_counter() - start)
            if model.classes_.tolist() != [0, 1]:
                raise ValueError(f"Orden inesperado de clases en fold {fold_id}: {model.classes_.tolist()}")
            self.models_.append(model)
        return self

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        fold_scores = [model.predict_proba(X)[:, 1] for model in self.models_]
        signal_scores = np.column_stack(fold_scores).mean(axis=1)
        return np.column_stack([1.0 - signal_scores, signal_scores])

    def predict_fold_scores(self, X: pd.DataFrame) -> pd.DataFrame:
        return pd.DataFrame(
            {f"score_fold_{idx}": model.predict_proba(X)[:, 1] for idx, model in enumerate(self.models_, start=1)}
        )

## Modelo final y regla de clasificación

Aquí se define el pipeline completo: preprocesamiento `P04`, ensamble de 5 modelos XGBoost y construcción de la submission. Cada modelo del ensamble se entrena sobre 4 folds y luego predice el conjunto de test; los cinco scores se promedian para obtener una estimación más estable por evento.

La clase final no se decide con un umbral estándar como 0.5 ni fijando manualmente un porcentaje de señales. Se usa el threshold absoluto `0.064...`, elegido sobre los scores out-of-fold del modelo porque fue la política con mayor AMS interno. En el test se etiqueta como `s` todo evento cuyo score promedio sea mayor o igual a ese valor. En la corrida de referencia esto seleccionó aproximadamente `14.07%` de los eventos como señal.


In [ ]:
class FinalHiggsPublicPipeline:
    def __init__(self) -> None:
        self.preprocessor = P04Preprocessor()
        self.model = CVXGBoostEnsemble(params=FINAL_XGB_PARAMS, n_splits=N_SPLITS, seed=SEED)

    def fit(self, train_df: pd.DataFrame) -> "FinalHiggsPublicPipeline":
        X_full = train_df[ALLOWED_FEATURE_COLUMNS].replace(MISSING_MARKER, np.nan)
        y_full = train_df["Label"].map({"b": 0, "s": 1}).astype(np.int8)
        weight_full = train_df["Weight"].astype(np.float64)
        train_idx, valid_idx = train_test_split(
            X_full.index.to_numpy(),
            test_size=VALID_SIZE,
            random_state=SEED,
            stratify=y_full,
        )

        X_train_raw = X_full.loc[train_idx].reset_index(drop=True)
        X_valid_raw = X_full.loc[valid_idx].reset_index(drop=True)
        y_train = y_full.loc[train_idx].reset_index(drop=True)
        y_valid = y_full.loc[valid_idx].reset_index(drop=True)
        w_train = weight_full.loc[train_idx].reset_index(drop=True)
        w_valid = weight_full.loc[valid_idx].reset_index(drop=True)

        self.preprocessor.fit(X_train_raw)
        X_train_processed = self.preprocessor.transform(X_train_raw)
        X_valid_processed = self.preprocessor.transform(X_valid_raw)
        X_model = pd.concat([X_train_processed, X_valid_processed], ignore_index=True)
        y_model = pd.concat([y_train, y_valid], ignore_index=True)
        w_model = pd.concat([w_train, w_valid], ignore_index=True)

        self.model.fit(X_model, y_model, sample_weight=w_model)
        self.metadata_ = {
            "X_train_processed_shape": list(X_train_processed.shape),
            "X_valid_processed_shape": list(X_valid_processed.shape),
            "X_model_shape": list(X_model.shape),
            "fit_seconds_per_fold": self.model.fit_seconds_,
            "fit_seconds_total": float(sum(self.model.fit_seconds_)),
        }
        return self

    def predict_scores(self, test_df: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray]:
        X_test_raw = test_df[ALLOWED_FEATURE_COLUMNS].replace(MISSING_MARKER, np.nan).reset_index(drop=True)
        X_test_processed = self.preprocessor.transform(X_test_raw)
        fold_scores = self.model.predict_fold_scores(X_test_processed)
        mean_scores = fold_scores.mean(axis=1).to_numpy(dtype=np.float64)
        score_frame = pd.concat(
            [test_df[["EventId"]].reset_index(drop=True).astype({"EventId": np.int64}), fold_scores],
            axis=1,
        )
        score_frame["score_mean"] = mean_scores
        self.metadata_["X_test_processed_shape"] = list(X_test_processed.shape)
        return score_frame, mean_scores


def threshold_classes(scores: np.ndarray, threshold: float) -> np.ndarray:
    classes = np.full(len(scores), "b", dtype=object)
    classes[scores >= threshold] = "s"
    return classes


def build_submission(event_id: pd.Series, scores: np.ndarray) -> pd.DataFrame:
    submission = pd.DataFrame(
        {
            "EventId": event_id.to_numpy(dtype=np.int64),
            "RankOrder": pd.Series(scores).rank(method="first", ascending=True).astype(np.int64),
            "Class": threshold_classes(scores, GLOBAL_OOF_OPTIMAL_THRESHOLD),
        }
    )[["EventId", "RankOrder", "Class"]]
    validate_submission(submission, event_id)
    return submission


def validate_submission(submission: pd.DataFrame, expected_event_id: pd.Series) -> None:
    if list(submission.columns) != ["EventId", "RankOrder", "Class"]:
        raise ValueError("Columnas inválidas para submission.")
    if len(submission) != EXPECTED_TEST_ROWS:
        raise ValueError(f"Cantidad de filas incorrecta: {len(submission)}")
    if not submission["EventId"].reset_index(drop=True).equals(expected_event_id.reset_index(drop=True)):
        raise ValueError("EventId no coincide con test.csv.")
    if submission["RankOrder"].nunique() != len(submission):
        raise ValueError("RankOrder no es único.")
    ranks = submission["RankOrder"].to_numpy(dtype=np.int64)
    if not np.array_equal(np.sort(ranks), np.arange(1, len(submission) + 1)):
        raise ValueError("RankOrder debe cubrir 1..n.")
    classes = set(submission["Class"].unique())
    if not classes <= {"s", "b"}:
        raise ValueError(f"Clases inválidas: {classes}")
    signal_count = int((submission["Class"] == "s").sum())
    if signal_count <= 0 or signal_count >= len(submission):
        raise ValueError(f"Signal count degenerado: {signal_count}")

## Carga y validación de los datos

Se localizan o descargan los archivos crudos, se leen `training.csv` y `test.csv`, y se verifica que tengan las columnas esperadas. Esta validación es importante porque el modelo final fue ajustado para la estructura oficial del desafío Higgs: 30 variables físicas de entrada, más `EventId`, `Weight` y `Label` en entrenamiento.

In [ ]:
work_dir = notebook_work_dir()
train_path, test_path = ensure_higgs_data(work_dir)
output_dir = work_dir

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
validate_raw_higgs(train_df, test_df)

summary = {
    "work_dir": ".",
    "train_path": str(train_path.relative_to(work_dir)),
    "test_path": str(test_path.relative_to(work_dir)),
    "train_shape": train_df.shape,
    "test_shape": test_df.shape,
    "pipeline": "P04",
    "model": "CV ensemble XGBoost SW04",
    "threshold_policy": f"global_oof_optimal_threshold={GLOBAL_OOF_OPTIMAL_THRESHOLD:.17f}",
    "reference_kaggle": REFERENCE_KAGGLE,
}
summary

## Entrenamiento y generación de archivos finales

En esta etapa se ajusta el preprocesamiento, se entrena el ensamble de XGBoost con todo el conjunto de entrenamiento disponible y se predicen scores para el test. A partir de esos scores se generan tres archivos en el mismo directorio del notebook.

`pipeline_final_taller_1.csv` es el archivo de submission. `pipeline_final_taller_1_scores.csv` guarda los scores de cada fold y el promedio. `pipeline_final_taller_1_metadata.json` guarda un resumen del modelo, los hiperparámetros y validaciones básicas de formato.

In [ ]:
pipeline = FinalHiggsPublicPipeline()
pipeline.fit(train_df)
scores_df, scores = pipeline.predict_scores(test_df)
submission_df = build_submission(test_df["EventId"].astype(np.int64), scores)

scores_path = output_dir / SCORES_FILENAME
submission_path = output_dir / SUBMISSION_FILENAME
metadata_path = output_dir / METADATA_FILENAME

scores_df.to_csv(scores_path, index=False)
submission_df.to_csv(submission_path, index=False)

metadata = {
    **summary,
    "hyperparameters": FINAL_XGB_PARAMS,
    "decision_threshold": GLOBAL_OOF_OPTIMAL_THRESHOLD,
    "validation": {
        "rows": int(len(submission_df)),
        "signal_count": int((submission_df["Class"] == "s").sum()),
        "background_count": int((submission_df["Class"] == "b").sum()),
        "signal_percent": float((submission_df["Class"] == "s").mean() * 100.0),
        "rank_order_unique": int(submission_df["RankOrder"].nunique()),
        "event_id_unique": int(submission_df["EventId"].nunique()),
    },
    "pipeline_metadata": pipeline.metadata_,
    "scores_path": str(scores_path.relative_to(work_dir)),
    "submission_path": str(submission_path.relative_to(work_dir)),
}
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False))

print(f"scores_path: {scores_path}")
print(f"submission_path: {submission_path}")
print(f"metadata_path: {metadata_path}")
metadata["validation"]

## Revisión final

Primero mostramos los scores generados por el ensamble, incluyendo los scores de cada fold y el promedio. Luego las primeras filas del archivo de submission con el formato esperado por Kaggle: `EventId`, `RankOrder` y `Class`. Por último, calculamos la proporción final de eventos clasificados como `s` y `b`; esa proporción queda determinada por el threshold global OOF y en la corrida de referencia fue cercana a `14.07%` de señales.


In [ ]:
display(scores_df.head())
display(submission_df.head())
submission_df["Class"].value_counts(normalize=True).rename("rate") * 100